# Atividade 2 — Pipeline RAG local com modelos abertos (Flan-T5 + FAISS)


Nesta atividade, damos continuidade ao estudo de Retrieval-Augmented Generation (RAG) iniciado na Atividade 1 focando independência de serviços pagos.

O pipeline construído nesta atividade utiliza:

- **PDF como fonte de conhecimento**;
- **Sentence-Transformers** para geração de embeddings;
- **FAISS** para indexação vetorial;
- **Flan-T5** como modelo de linguagem instruído;
- **LangChain** apenas como camada de orquestração, sem dependência de APIs proprietárias.

Ao final, o notebook implementa um **sistema RAG interativo**, no qual o usuário pode formular perguntas em um loop de diálogo, recebendo respostas fundamentadas em trechos recuperados do documento.

<br>

## Objetivos da atividade

Ao concluir esta atividade, espera-se que o estudante seja capaz de:

- compreender o papel de cada etapa de um pipeline RAG completo;
- implementar um sistema RAG **reprodutível e modular**;
- distinguir claramente entre:
  - pré-processamento,
  - indexação,
  - recuperação,
  - geração;
- lidar com **limitações reais de modelos** (ex.: tamanho máximo de contexto);
- justificar decisões de projeto em sistemas de PLN aplicados.

<br>

> A execução completa do sistema ocorre **exclusivamente na Seção 7**, reforçando a distinção entre *definir* e *executar*.


## 1. Instalação e organização do ambiente

Nesta atividade, construiremos um pipeline RAG (Retrieval-Augmented Generation) **sem uso de API paga** e **sem GPU**.
A arquitetura utiliza exclusivamente bibliotecas gratuitas e executa em CPU no Google Colab.

Componentes do pipeline:

- **PDF (extração)**: `pypdf`
- **Chunking (segmentação)**: `langchain-text-splitters`
- **Embeddings**: `sentence-transformers` (Hugging Face)
- **Índice vetorial**: `faiss-cpu`
- **LLM em CPU**: `google/flan-t5-base` via `transformers`
- **Orquestração**: `langchain` (módulos comunitários)

Observação: o Flan-T5 possui limite curto de contexto; por isso, mais adiante controlaremos explicitamente o **orçamento de tokens** do contexto recuperado.


In [44]:
!pip -q install -U \
  pypdf \
  langchain \
  langchain-community \
  langchain-text-splitters \
  sentence-transformers \
  faiss-cpu \
  transformers \
  accelerate

### 1.1. Imports e configuração central

Nesta subseção:

1. Importamos bibliotecas (agrupadas por finalidade);
2. Definimos uma única estrutura de configuração (`RAGConfig`) para controlar:
   - chunking,
   - modelos,
   - cache do índice,
   - parâmetros de geração.

Essa centralização evita redundância e facilita a manutenção do notebook.


In [56]:
# =========================
# Imports — Python (stdlib)
# =========================
import os
import re
import json
import pickle
import hashlib
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

# =========================
# Imports — Colab/Drive
# =========================
from google.colab import drive

# =========================
# Imports — PDF
# =========================
from pypdf import PdfReader

# =========================
# Imports — LangChain
# =========================
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline

# =========================
# Imports — Transformers (Flan-T5)
# =========================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# =========================
# Configuração central
# =========================
@dataclass
class RAGConfig:
    # Drive
    drive_root: str = "/content/drive/MyDrive"

    # Cache do índice (padrão do notebook Atividade 1)
    # "fixed": salva em uma pasta fixa (fixed_cache_dir)
    # "pdf_dir": salva ao lado do PDF
    cache_dir_mode: str = "pdf_dir"
    fixed_cache_dir: str = "/content/drive/MyDrive/rag_indexes"

    # Chunking (caracteres, compatível com RecursiveCharacterTextSplitter)
    chunk_size: int = 500
    chunk_overlap: int = 80
    separators: Optional[List[str]] = None

    # Embeddings e LLM (CPU)
    emb_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model_name: str = "google/flan-t5-base"

    # Recuperação / contexto
    top_k: int = 4

    # Controle de tokens (importante para Flan-T5)
    max_input_tokens: int = 512
    reserved_for_prompt_tokens: int = 180

    # Geração
    max_new_tokens: int = 256
    do_sample: bool = False

### 1.2. Checagem rápida de ambiente (CPU)

In [28]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível?", torch.cuda.is_available())
print("Dispositivo esperado:", "CPU" if not torch.cuda.is_available() else "GPU (não necessário)")

PyTorch: 2.9.0+cpu
CUDA disponível? False
Dispositivo esperado: CPU


## 2. Acesso ao Drive e preparação do corpus (definições)

Nesta seção são definidas as funções utilitárias responsáveis por:

- montar o Google Drive no ambiente Colab;
- resolver caminhos relativos ao `MyDrive`;
- extrair texto de arquivos PDF página a página;
- associar metadados (arquivo e número da página);
- aplicar uma limpeza mínima ao texto extraído.


In [46]:
def mount_drive(force: bool = False) -> None:
    """
    Monta o Google Drive no ambiente Google Colab.

    Parâmetros
    ----------
    force : bool
        Se True, força a remontagem do Drive.
    """
    from google.colab import drive
    drive.mount("/content/drive", force_remount=force)


def resolve_drive_path(relative_path: str, base: str) -> str:
    """
    Constrói o caminho absoluto a partir de um caminho relativo ao MyDrive
    e valida a existência do arquivo.
    """
    import os

    full_path = os.path.join(base, relative_path)
    if not os.path.isfile(full_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {full_path}")
    return full_path


### Extração de texto do PDF com metadados

A extração é realizada **página a página**, o que permite:

- associar cada trecho ao número da página;
- rastrear evidências durante a recuperação;
- justificar respostas produzidas pelo modelo (RAG explicável).

Os metadados adotados são:
- `source`: nome do arquivo PDF;
- `page`: número da página (1-indexado).


In [47]:
def minimal_cleanup(text: str) -> str:
    """
    Aplica uma limpeza mínima ao texto extraído do PDF:
    - normaliza espaços;
    - remove excesso de quebras de linha.

    Essa limpeza melhora a segmentação e a recuperação,
    sem descaracterizar o texto original.
    """
    import re

    text = text.replace("\t", " ")
    text = re.sub(r"[ \u00A0]+", " ", text)   # inclui NBSP
    text = re.sub(r"\n{3,}", "\n\n", text)    # reduz linhas em branco
    return text.strip()


def extract_pages_with_metadata(pdf_path: str):
    """
    Extrai texto do PDF página a página e associa metadados.

    Parâmetros
    ----------
    pdf_path : str
        Caminho absoluto do arquivo PDF.

    Retorna
    -------
    List[Dict]
        Lista no formato:
        {
          "text": str,
          "metadata": {
              "source": str,
              "page": int
          }
        }
    """
    import os
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    source = os.path.basename(pdf_path)

    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append({
            "text": minimal_cleanup(text),
            "metadata": {
                "source": source,
                "page": page_number
            }
        })

    return pages

### Saída esperada (conceitual)

As funções definidas nesta seção permitem que, na **Seção 7**, o código produza:

- `pdf_path`: caminho absoluto do PDF;
- `pages`: lista de páginas com texto e metadados, no formato:

```python
{
  "text": "...",
  "metadata": {
      "source": "documento.pdf",
      "page": 3
  }
}


## 3. Chunking e construção dos documentos (definições)

Nesta seção definimos as funções responsáveis por transformar o texto extraído
do PDF (página a página) em **unidades semânticas menores**, chamadas *chunks*.

Esses chunks são encapsulados em objetos `Document`, que combinam:

- `page_content`: o texto do chunk;
- `metadata`: informações de origem (arquivo e página).

Essa etapa é fundamental em sistemas RAG, pois:

- controla a granularidade da recuperação;
- influencia diretamente a qualidade da busca vetorial;
- impacta custo computacional e limite de contexto do modelo.


### Estratégia de segmentação adotada

Utilizamos o `RecursiveCharacterTextSplitter`, que aplica uma estratégia
hierárquica de separação:

1. tenta dividir por parágrafos;
2. depois por quebras de linha;
3. depois por frases;
4. por fim, por espaços ou caracteres individuais.

Os parâmetros principais são:

- `chunk_size`: tamanho máximo do chunk (em caracteres);
- `chunk_overlap`: sobreposição entre chunks consecutivos.

Esses parâmetros serão configurados via `RAGConfig` e utilizados apenas
no momento da execução.


In [49]:
def build_documents_from_pages(
    pages,
    chunk_size: int,
    chunk_overlap: int,
    separators=None
):
    """
    Constrói uma lista de objetos Document a partir de páginas extraídas do PDF,
    aplicando chunking e preservando metadados.

    Parâmetros
    ----------
    pages : List[Dict]
        Lista no formato:
        {
          "text": str,
          "metadata": {
              "source": str,
              "page": int
          }
        }

    chunk_size : int
        Tamanho máximo de cada chunk (em caracteres).

    chunk_overlap : int
        Número de caracteres de sobreposição entre chunks consecutivos.

    separators : list, opcional
        Lista de separadores usados pelo RecursiveCharacterTextSplitter.
        Se None, utiliza a configuração padrão.

    Retorna
    -------
    List[Document]
        Lista de documentos prontos para indexação vetorial.
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    if separators is None:
        separators = ["\n\n", "\n", "  ", ". ", " ", ""]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators
    )

    documents = []

    for p in pages:
        splits = splitter.split_text(p["text"])
        for chunk in splits:
            documents.append(
                Document(
                    page_content=chunk,
                    metadata=p["metadata"]
                )
            )

    return documents

### Saída esperada (conceitual)

As funções definidas nesta seção permitem que, na **Seção 7**, o código produza:

- `documents`: lista de objetos `Document`, no formato:

```python
Document(
    page_content="trecho de texto...",
    metadata={
        "source": "arquivo.pdf",
        "page": 5
    }
)


## 4. Cache do índice FAISS e fingerprint (definições)

Nesta seção definimos a infraestrutura responsável por **persistir e reutilizar**
o índice vetorial FAISS associado a um PDF.

A persistência é essencial em pipelines RAG porque:
- a construção do índice é a etapa mais custosa;
- o mesmo documento costuma ser consultado várias vezes;
- em contexto didático, evita recomputações desnecessárias a cada execução.

A estratégia adotada é a mesma da Atividade 1:

1. Cada PDF possui um diretório de índice próprio;
2. Um **fingerprint** registra:
   - o hash do PDF (SHA-256);
   - parâmetros relevantes do pipeline (chunking e embeddings);
3. O índice só é reutilizado se o fingerprint atual for idêntico ao salvo.


### Hash do PDF

O hash SHA-256 é utilizado para detectar alterações no conteúdo do arquivo PDF.
Mesmo pequenas modificações (ou substituição do arquivo mantendo o nome)
resultam em um hash diferente, forçando a reconstrução do índice.


In [50]:
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    """
    Calcula o hash SHA-256 de um arquivo em modo streaming.

    Parâmetros
    ----------
    path : str
        Caminho do arquivo.
    chunk_size : int
        Tamanho do bloco de leitura (bytes).

    Retorna
    -------
    str
        Hash SHA-256 do arquivo.
    """
    import hashlib

    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

### Diretório do índice RAG

Cada PDF possui um diretório de índice próprio, com o nome:   
`<nome_do_pdf>_rag_index/`


Esse diretório pode ser criado:
- ao lado do PDF (modo `pdf_dir`);
- ou em uma pasta fixa no Drive (modo `fixed`).

Essa escolha é controlada pela configuração central (`RAGConfig`).


In [51]:
def get_index_dir(pdf_path: str, config) -> str:
    """
    Determina e cria o diretório do índice RAG associado a um PDF.

    Parâmetros
    ----------
    pdf_path : str
        Caminho absoluto do PDF.
    config : RAGConfig
        Configuração central do pipeline.

    Retorna
    -------
    str
        Caminho do diretório do índice.
    """
    import os

    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]

    if config.cache_dir_mode == "fixed":
        os.makedirs(config.fixed_cache_dir, exist_ok=True)
        d = os.path.join(config.fixed_cache_dir, f"{pdf_name}_rag_index")
        os.makedirs(d, exist_ok=True)
        return d

    pdf_dir = os.path.dirname(pdf_path)
    d = os.path.join(pdf_dir, f"{pdf_name}_rag_index")
    os.makedirs(d, exist_ok=True)
    return d

### Fingerprint do índice

O fingerprint representa o “estado” do índice no momento em que foi criado.
Se qualquer elemento do fingerprint mudar, o índice deve ser reconstruído.

Elementos considerados:
- hash do PDF;
- modelo de embeddings;
- parâmetros de chunking.


In [52]:
def make_fingerprint(pdf_path: str, config) -> dict:
    """
    Gera o fingerprint do índice a partir do PDF e da configuração.
    """
    return {
        "pdf_path": pdf_path,
        "pdf_sha256": sha256_file(pdf_path),
        "emb_model_name": config.emb_model_name,
        "chunk_size": config.chunk_size,
        "chunk_overlap": config.chunk_overlap,
        "separators": config.separators,
    }


def cache_paths(index_dir: str) -> dict:
    """
    Define os caminhos dos arquivos de cache do índice FAISS.
    """
    import os

    return {
        "faiss": os.path.join(index_dir, "index.faiss"),
        "store": os.path.join(index_dir, "index.pkl"),
        "cfg": os.path.join(index_dir, "config.json"),
    }


def cache_exists(index_dir: str) -> bool:
    """
    Verifica se todos os arquivos esperados do cache existem.
    """
    p = cache_paths(index_dir)
    return all(os.path.isfile(p[k]) for k in p)

### Persistência do fingerprint

O fingerprint é armazenado em um arquivo `config.json` dentro do diretório
do índice. Ele é utilizado para decidir se o índice pode ser reutilizado
ou deve ser reconstruído.


In [53]:
def save_cache_config(index_dir: str, fingerprint: dict, faiss_ntotal: int) -> None:
    """
    Salva o fingerprint e metadados do índice em config.json.
    """
    import json
    from datetime import datetime

    p = cache_paths(index_dir)
    cfg = {
        "fingerprint": fingerprint,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "faiss_ntotal": int(faiss_ntotal),
    }
    with open(p["cfg"], "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)


def load_cache_config(index_dir: str) -> dict:
    """
    Carrega o config.json do índice.
    """
    import json

    p = cache_paths(index_dir)
    with open(p["cfg"], "r", encoding="utf-8") as f:
        return json.load(f)

### Construção ou carregamento do índice FAISS

A função a seguir encapsula a lógica:

- se existir cache **e** o fingerprint conferir → carrega o índice;
- caso contrário → reconstrói o índice e atualiza o cache.


In [54]:
def build_or_load_vectorstore_with_fingerprint(
    cfg,
    pdf_path: str,
    documents,
    embeddings
):
    """
    Constrói ou carrega um índice FAISS com base no fingerprint.
    """
    from langchain_community.vectorstores import FAISS

    index_dir = get_index_dir(pdf_path, cfg)
    fp_now = make_fingerprint(pdf_path, cfg)

    if cache_exists(index_dir):
        try:
            cfg_saved = load_cache_config(index_dir)
            if cfg_saved.get("fingerprint") == fp_now:
                return FAISS.load_local(
                    index_dir,
                    embeddings,
                    allow_dangerous_deserialization=True
                )
        except Exception:
            pass  # força reconstrução

    # Reconstrói índice
    vs = FAISS.from_documents(documents=documents, embedding=embeddings)
    vs.save_local(index_dir)

    try:
        ntotal = int(vs.index.ntotal)
    except Exception:
        ntotal = -1

    save_cache_config(index_dir, fp_now, ntotal)
    return vs

### Saída conceitual desta seção

As funções definidas nesta seção permitem que, na **Seção 7**, o pipeline:

- reutilize índices FAISS sempre que possível;
- detecte automaticamente mudanças no PDF ou na configuração;
- mantenha índices organizados por documento.


## 5. Modelos, embeddings e controle de tokens (definições)

Nesta seção definimos as funções responsáveis por:

1. Inicializar o modelo de embeddings (Sentence-Transformers);
2. Inicializar o modelo de linguagem (Flan-T5 em CPU);
3. Controlar explicitamente o **orçamento de tokens** do contexto enviado ao modelo.

Essas definições são essenciais porque:

- o Flan-T5 possui um limite curto de tokens de entrada (≈ 512);
- em RAG, o contexto recuperado pode facilmente ultrapassar esse limite;
- o controle explícito de tokens evita erros e torna o pipeline robusto.


### Modelo de embeddings

Utilizamos um modelo da família **Sentence-Transformers**, amplamente empregado
em tarefas de busca semântica e RAG.

Características do modelo adotado (`all-MiniLM-L6-v2`):

- gratuito e de código aberto;
- leve, adequado para execução em CPU;
- bom compromisso entre custo computacional e qualidade semântica.

O modelo é carregado apenas uma vez durante a execução do pipeline.


In [58]:
def normalize_text(s: str) -> str:
    s = s.lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s

class NormalizedEmbeddings(Embeddings):
    """
    Wrapper compatível com LangChain:
      - embed_documents(list[str]) -> list[list[float]]
      - embed_query(str) -> list[float]
    Inclui __call__ como fallback (alguns vectorstores tentam chamar o objeto).
    """
    def __init__(self, base: Embeddings):
        self.base = base

    def embed_documents(self, texts):
        return self.base.embed_documents([normalize_text(t) for t in texts])

    def embed_query(self, text):
        return self.base.embed_query(normalize_text(text))

    def __call__(self, text):
        return self.embed_query(text)

def build_embeddings(config):
    """
    Inicializa o modelo de embeddings a partir da configuração central.

    Parâmetros
    ----------
    config : RAGConfig
        Configuração do pipeline.

    Retorna
    -------
    HuggingFaceEmbeddings
        Objeto de embeddings para uso no FAISS.
    """
    base = HuggingFaceEmbeddings(model_name=config.emb_model_name)
    return NormalizedEmbeddings(base)

### Modelo de linguagem (LLM): Flan-T5 em CPU

O modelo de linguagem utilizado é o **Flan-T5**, executado localmente em CPU.

Motivações da escolha:

- modelo instruído (instruction-tuned);
- não depende de API paga;
- compatível com ambientes sem GPU;
- adequado para fins didáticos e experimentais.

Devido ao limite de tokens do encoder do Flan-T5, o controle de contexto
é tratado explicitamente na próxima subseção.


In [59]:
def build_llm(config):
    """
    Inicializa o modelo Flan-T5 para geração de texto em CPU.

    Parâmetros
    ----------
    config : RAGConfig
        Configuração do pipeline.

    Retorna
    -------
    HuggingFacePipeline
        Wrapper do LangChain para o modelo Flan-T5.
    """
    tokenizer = AutoTokenizer.from_pretrained(config.llm_model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(config.llm_model_name)

    gen_pipeline = pipeline(
        task="text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=config.max_new_tokens,
        do_sample=config.do_sample,
        truncation=True,
        max_length=config.max_input_tokens,
    )

    return HuggingFacePipeline(pipeline=gen_pipeline)

### Controle explícito do orçamento de tokens

Em modelos baseados em *encoder-decoder* (como o Flan-T5), o limite de tokens
da entrada é rígido. Por isso:

- o contexto recuperado pelo FAISS deve ser truncado **por tokens**, não por caracteres;
- parte do orçamento de tokens deve ser reservada para instruções e pergunta.

A função a seguir implementa esse controle de forma explícita e reutilizável.


In [60]:
def format_context_from_docs_token_budget(
    docs,
    tokenizer,
    max_input_tokens: int,
    reserved_for_prompt_tokens: int
) -> str:
    """
    Monta um contexto textual respeitando um orçamento máximo de tokens.

    Parâmetros
    ----------
    docs : List[Document]
        Documentos recuperados pelo FAISS.
    tokenizer :
        Tokenizer do modelo de linguagem.
    max_input_tokens : int
        Limite máximo de tokens aceitos pelo modelo.
    reserved_for_prompt_tokens : int
        Tokens reservados para instruções e pergunta.

    Retorna
    -------
    str
        Contexto truncado por tokens, seguro para envio ao modelo.
    """
    budget = max(1, max_input_tokens - reserved_for_prompt_tokens)

    parts = []
    for d in docs:
        source = d.metadata.get("source", "desconhecido")
        page = d.metadata.get("page", "?")
        header = f"[Fonte: {source} | Página: {page}]"
        body = d.page_content.strip()
        parts.append(header + "\n" + body)

    full_context = "\n\n".join(parts)

    enc = tokenizer(
        full_context,
        add_special_tokens=False,
        truncation=True,
        max_length=budget,
        return_tensors=None
    )

    # enc["input_ids"] é uma lista de ids (não tensor) quando return_tensors=None
    safe_context = tokenizer.decode(enc["input_ids"], skip_special_tokens=True)
    return safe_context

### Papel desta seção no pipeline

As funções definidas nesta seção serão utilizadas para:

- inicializar embeddings e LLM (uma única vez);
- garantir que o contexto enviado ao modelo respeite os limites de tokens;
- evitar erros de execução e respostas inconsistentes.


## 6) Definição do pipeline RAG

Nesta seção definimos a classe `RAGPipeline`, responsável por **orquestrar**
todas as etapas do sistema RAG descritas nas seções anteriores, mantendo
uma separação clara entre **definição** e **execução** do pipeline.

As responsabilidades integradas pela classe incluem:

- acesso ao Google Drive e resolução do caminho do PDF (Seção 2);
- extração de texto com metadados (arquivo e número da página) (Seção 2);
- segmentação (*chunking*) e construção de documentos semânticos (Seção 3);
- persistência e reutilização do índice vetorial FAISS, com fingerprint (Seção 4);
- inicialização dos modelos e controle explícito de orçamento de tokens (Seção 5).

⚠️ Importante: nesta seção **nenhuma etapa do pipeline é executada**.
A classe apenas define **como** as etapas se conectam.
A execução completa ocorre exclusivamente na **Seção 7**.

<br>

### Onde ocorre o RAG neste pipeline

É fundamental destacar que, neste notebook, **o RAG não está no prompt**.
O RAG é um **processo em duas etapas**, claramente separadas dentro do método
`ask()` da classe `RAGPipeline`:

**1. Retrieval — a etapa “R” do RAG**

A etapa de *retrieval* ocorre **antes da criação do prompt**, quando a pergunta
do usuário é utilizada para recuperar trechos relevantes do documento indexado.

Conceitualmente, essa etapa corresponde à chamada de busca vetorial no FAISS,
que compara a pergunta do usuário com os embeddings dos *chunks* do documento
e retorna os trechos semanticamente mais próximos.

Nesta fase:

- a pergunta do usuário é transformada internamente em embedding;
- o índice FAISS realiza a busca por similaridade;
- são recuperados os trechos que servirão de **base factual** para a resposta.

Sem essa etapa, o sistema deixa de ser um RAG e passa a ser apenas um
modelo de linguagem respondendo com base em conhecimento paramétrico.

**2. Construção do contexto (ponte entre Retrieval e Generation)**

Os trechos recuperados não são enviados diretamente ao modelo.
Eles passam por uma etapa intermediária de organização e truncamento,
na qual é construído um **contexto textual explícito**, respeitando o
limite de tokens do modelo de linguagem.

Essa etapa transforma o resultado da recuperação vetorial em um texto
que pode ser consumido de forma segura pelo modelo.

**3. Generation — a etapa “G” do RAG**

Somente após a recuperação e a construção do contexto é que o prompt é criado
e enviado ao modelo de linguagem.

Nesta etapa, o modelo:

- **não realiza busca**;
- **não tem acesso direto ao PDF**;
- responde exclusivamente com base no contexto fornecido.

<br>

### Síntese conceitual

Neste pipeline, o fluxo completo do RAG pode ser resumido como:

Pergunta do usuário  
↓  
Recuperação vetorial (FAISS) ←── R do RAG  
↓  
Construção do contexto textual  
↓  
Modelo de linguagem (Flan-T5) ←── G do RAG  
↓  
Resposta final  


In [61]:
class RAGPipeline:
    """
    Esta classe organiza todas as etapas do pipeline, mas não executa
    nada automaticamente. Os métodos devem ser chamados explicitamente
    na Seção 7.
    """

    def __init__(self, config):
        """
        Inicializa o pipeline com a configuração central.

        Parâmetros
        ----------
        config : RAGConfig
            Configuração do pipeline.
        """
        self.cfg = config

        # Componentes do pipeline (inicializados em prepare)
        self.embeddings = None
        self.llm = None
        self.vectorstore = None
        self.pdf_path = None

    def prepare(self, relative_pdf_path: str, force_remount_drive: bool = False) -> None:
        """
        Prepara o pipeline RAG:
        - monta o Drive;
        - resolve o caminho do PDF;
        - extrai páginas com metadados;
        - realiza chunking;
        - constrói ou carrega o índice FAISS.

        Parâmetros
        ----------
        relative_pdf_path : str
            Caminho relativo ao MyDrive do PDF.
        force_remount_drive : bool
            Se True, força a remontagem do Google Drive.
        """
        # Seção 2
        mount_drive(force=force_remount_drive)
        self.pdf_path = resolve_drive_path(
            relative_pdf_path,
            base=self.cfg.drive_root
        )

        pages = extract_pages_with_metadata(self.pdf_path)

        # Seção 3
        documents = build_documents_from_pages(
            pages=pages,
            chunk_size=self.cfg.chunk_size,
            chunk_overlap=self.cfg.chunk_overlap,
            separators=self.cfg.separators
        )

        # Seção 5
        self.embeddings = build_embeddings(self.cfg)
        self.llm = build_llm(self.cfg)

        # Seção 4
        self.vectorstore = build_or_load_vectorstore_with_fingerprint(
            cfg=self.cfg,
            pdf_path=self.pdf_path,
            documents=documents,
            embeddings=self.embeddings
        )

    def ask(self, question: str, k: int = None):
      """
      Executa uma consulta RAG (Retrieval + Generation).

      Retorna:
        - answer (str)
        - docs (List[Document])
        - scores (List[float])
      """
      if self.vectorstore is None or self.llm is None:
          raise RuntimeError("Pipeline não preparado. Execute prepare(...) antes de ask(...).")

      k = k or self.cfg.top_k

      # =========================================================
      # (R) RETRIEVAL — Recuperação vetorial (FAISS)
      # =========================================================
      # Aqui ocorre o "R" do RAG.
      # Usamos similarity_search_with_score para:
      # - obter relevância dos trechos
      # - deduplicar páginas repetidas
      # - reduzir ruído no contexto enviado ao LLM

      docs_scores = self.vectorstore.similarity_search_with_score(question, k=k)

      seen = set()
      filtered = []
      for d, s in docs_scores:
          key = (d.metadata.get("source"), d.metadata.get("page"))
          if key in seen:
              continue
          seen.add(key)
          filtered.append((d, s))

      docs = [d for d, s in filtered]
      scores = [s for d, s in filtered]

      # =========================================================
      # Construção do CONTEXTO (ponte entre R e G)
      # =========================================================
      # Os trechos recuperados são organizados em um contexto textual
      # explícito, com indicação de fonte e página, respeitando o
      # orçamento de tokens do modelo (Flan-T5).
      #
      # Aqui ocorre a transformação do resultado da recuperação
      # vetorial em texto consumível pelo modelo de linguagem.
      tokenizer = self.llm.pipeline.tokenizer
      context = format_context_from_docs_token_budget(
          docs=docs,
          tokenizer=tokenizer,
          max_input_tokens=self.cfg.max_input_tokens,
          reserved_for_prompt_tokens=self.cfg.reserved_for_prompt_tokens
      )

      # =========================================================
      # (G) GENERATION — Geração da resposta
      # =========================================================
      # Nesta etapa ocorre o "G" do RAG.
      # O modelo de linguagem NÃO realiza busca e NÃO tem acesso
      # direto ao PDF. Ele apenas recebe o contexto recuperado
      # e gera a resposta com base nesse material.
      prompt = f"""Você é um assistente acadêmico. Responda em português.
Use APENAS o contexto abaixo.
Não repita a pergunta e não copie o contexto literalmente; extraia a informação e responda diretamente.
Se não houver informação suficiente no contexto, responda: "Não encontrei essa informação no documento."

PERGUNTA:
{question}

CONTEXTO:
{context}

RESPOSTA (objetiva e bem estruturada):
"""

      # Tokenização com truncamento forte (garante <= max_input_tokens)
      inputs = tokenizer(
          prompt,
          truncation=True,
          max_length=self.cfg.max_input_tokens,
          return_tensors="pt"
      )

      # Geração diretamente pelo modelo (evita retokenização do pipeline)
      model = self.llm.pipeline.model

      gen_ids = model.generate(
          **inputs,
          max_new_tokens=self.cfg.max_new_tokens,
          do_sample=self.cfg.do_sample,
      )

      answer = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

      return answer, docs, scores

### Papel desta seção no notebook

A classe `RAGPipeline` atua como o **núcleo de integração** do sistema.

Até este ponto, o notebook contém apenas:
- definições de funções;
- definições de classes;
- explicações conceituais.

A **Seção 7** será responsável por:
- instanciar `RAGConfig`;
- criar um objeto `RAGPipeline`;
- executar `prepare(...)`;
- executar uma ou mais chamadas a `ask(...)`.

Isso garante clareza, reprodutibilidade e boa organização didática.


## 7. Execução do pipeline e chat interativo (estilo Atividade 1)

Nesta seção executamos o sistema completo:

1. Instanciamos a configuração (`RAGConfig`);
2. Preparamos o pipeline (`prepare`):
   - monta o Drive;
   - resolve o PDF;
   - extrai texto + metadados;
   - faz chunking;
   - constrói ou carrega o índice FAISS (com fingerprint);
3. Iniciamos um **chat interativo**:
   - o usuário digita perguntas no terminal;
   - o sistema recupera trechos relevantes (com fonte/página);
   - o LLM (Flan-T5) produz a resposta baseada no contexto recuperado.

Para sair, digite: `sair`, `exit`, `quit` ou `q`.


### Preparação do pipeline (executa uma única vez)

In [62]:
# Ajuste o caminho relativo ao MyDrive (o mesmo padrão das atividades anteriores)
relative_pdf_path = "fatec/jacarei/DSM/PLN/atividades/Atividade 2/disciplinas-modalidade.pdf"

# 1) Configuração
cfg = RAGConfig(
    # você pode ajustar aqui se desejar
    top_k=2,
    chunk_size=500,
    chunk_overlap=80,
    max_input_tokens=512,
    reserved_for_prompt_tokens=180,
    max_new_tokens=256,
    do_sample=False,
    cache_dir_mode="pdf_dir",  # ou "fixed"
    fixed_cache_dir="/content/drive/MyDrive/rag_indexes"
)

# 2) Pipeline
pipeline_rag = RAGPipeline(cfg)

# 3) Preparação (monta drive, extrai, indexa/carrega)
pipeline_rag.prepare(relative_pdf_path)

print("Sistema pronto para perguntas.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Device set to use cpu


Sistema pronto para perguntas.


### Função de chat interativo (loop de perguntas)

In [63]:
def interactive_chat(pipeline: RAGPipeline, show_sources: bool = True) -> None:
    """
    Loop interativo no estilo da Atividade 1.
    - Lê perguntas do usuário (input)
    - Recupera evidências (FAISS)
    - Gera resposta (Flan-T5)
    """
    EXIT = {"sair", "exit", "quit", "q"}
    print("Sistema pronto. Digite perguntas. Para sair, digite 'sair'.")
    print("-" * 60)

    while True:
        print("\n(Aguardando pergunta... digite e pressione Enter)")
        q = input("Você: ").strip()

        if not q:
            continue
        if q.lower() in EXIT:
            print("Encerrando.")
            break

        # Recupera + responde (o ask já retorna docs recuperados)
        answer_text, docs, scores = pipeline.ask(q)

        if show_sources:
            print("\nFontes recuperadas:")
            for i, (d, s) in enumerate(zip(docs, scores), start=1):
                src = d.metadata.get("source", "desconhecido")
                page = d.metadata.get("page", "?")
                page_str = f"p.{page}" if isinstance(page, int) else "p.?"
                print(f"- [Fonte {i}] {src} | {page_str} | score={s:.3f}")

        print("\nAssistente:", answer_text)
        print("-" * 60)

### Iniciar o chat

In [64]:
interactive_chat(pipeline_rag, show_sources=True)

Sistema pronto. Digite perguntas. Para sair, digite 'sair'.
------------------------------------------------------------

(Aguardando pergunta... digite e pressione Enter)
Você: qual é o semestre que está a disciplina de algebra?

Fontes recuperadas:
- [Fonte 1] disciplinas-modalidade.pdf | p.3 | score=1.398
- [Fonte 2] disciplinas-modalidade.pdf | p.1 | score=1.457

Assistente: ING-088 Inglés IV Remota - - 40 40 Total de aulas semestrais - 20 460 480 Total de aulas do curso 360 1980 540 2880 [Fonte: disciplinas-modalidade.pdf | Page 1] Banco de Dados – Relacional Estrutura de Dados Terceiro semestre Técnicas de Programaço II Desenvolvimento Web III lgebra Linear Gesto gil de Projetos de Software Banco de Dados - No Relacional Interaço Humano Computador Inglés I Quarto semestre Integraço e Entrega Contnua Laboratório de Desenvolvimento Web Internet das Coisas e Aplicaçes Programaço para Dispositivos Móveis I RESPOSTA (objectiva e bem estruturada):
--------------------------------------